# Exerkine Nucleotide Sequence Gym
A nucleotide-only counterpart to the protein Gym. It designs **protein-preserving exerkine CDS/mRNA sequences** (synonymous codon design under sequence constraints) and validates every design against **NCBI** (RefSeq mRNA, BLASTn), **UniProt** (translated protein) and the **ExerkineAtlas**.

Per design:
1. **Fidelity**: translates exactly to the NCBI RefSeq protein and matches the UniProt entry
2. **Nucleotide novelty**: identity to the human RefSeq CDS, the rat ortholog CDS, and top NCBI BLASTn hit
3. **Sequence quality**: GC, codon adaptation (CAI), homopolymers, forbidden motifs
4. **RNA structure**: ViennaRNA MFE (full CDS and 5' window as an initiation-accessibility proxy)
5. **Optional**: Nucleotide Transformer masked log-likelihood
6. **Atlas grounding**: gene is annotated in the ExerkineAtlas (you supply the export)

In [1]:
!pip -q install biopython pandas numpy requests ViennaRNA torch transformers

In [3]:
import io, time, random, requests, numpy as np, pandas as pd
from collections import Counter, defaultdict
from itertools import groupby
from Bio import SeqIO, Entrez
from Bio.Seq import Seq
from Bio.Data import CodonTable
from Bio.Align import PairwiseAligner, substitution_matrices
from Bio.Blast import NCBIWWW, NCBIXML
import RNA

Entrez.email = "sfdanielgomez@gmail.com"   # <-- set your email (NCBI requirement)
HUMAN, RAT = "Homo sapiens", "Rattus norvegicus"
GENES = ["IL6", "BDNF", "IL15", "FGF21", "GDF15", "VEGFA", "FNDC5", "METRNL"]  # edit freely

nt_aln = PairwiseAligner(); nt_aln.mode = "global"
nt_aln.match_score, nt_aln.mismatch_score = 1, -1
nt_aln.open_gap_score, nt_aln.extend_gap_score = -5, -1
aa_aln = PairwiseAligner(); aa_aln.mode = "global"
aa_aln.substitution_matrix = substitution_matrices.load("BLOSUM62")
aa_aln.open_gap_score, aa_aln.extend_gap_score = -10, -0.5

def identity(a, b, aligner):
    aln = aligner.align(a, b)[0]
    m = sum(a[i] == b[j] for (s1, e1), (s2, e2) in zip(*aln.aligned)
            for i, j in zip(range(s1, e1), range(s2, e2)))
    return m / min(len(a), len(b))

## 1. Seeds: RefSeq CDS from NCBI, protein from UniProt

In [4]:
def ncbi_cds(gene, organism):
    term = f"{gene}[Gene Name] AND {organism}[Organism] AND srcdb_refseq_known[Properties] AND biomol_mrna[Properties]"
    ids = Entrez.read(Entrez.esearch(db="nuccore", term=term, retmax=1))["IdList"]
    time.sleep(0.4)
    if not ids: return None
    rec = SeqIO.read(Entrez.efetch(db="nuccore", id=ids[0], rettype="gb", retmode="text"), "genbank")
    time.sleep(0.4)
    for f in rec.features:
        if f.type == "CDS":
            return dict(acc=rec.id, gene=gene, cds=str(f.extract(rec.seq)).upper(),
                        protein=f.qualifiers.get("translation", [""])[0])
    return None

def uniprot_protein(gene, organism_id=9606):
    q = f"(gene_exact:{gene}) AND (organism_id:{organism_id}) AND (reviewed:true)"
    r = requests.get("https://rest.uniprot.org/uniprotkb/search",
                     params={"query": q, "format": "fasta", "size": 1}, timeout=30)
    recs = list(SeqIO.parse(io.StringIO(r.text), "fasta"))
    return (recs[0].id.split("|")[1], str(recs[0].seq)) if recs else (None, None)

human = {g: ncbi_cds(g, HUMAN) for g in GENES}
rat   = {g: ncbi_cds(g, RAT) for g in GENES}
human = {g: v for g, v in human.items() if v}
pd.DataFrame([(g, v["acc"], len(v["cds"]), len(v["protein"])) for g, v in human.items()],
             columns=["gene", "refseq", "cds_nt", "protein_aa"])

,gene,refseq,cds_nt,protein_aa
0,IL6,NM_001371096.1,570,189
1,BDNF,NM_170734.4,789,262
2,IL15,NM_172175.3,408,135
3,FGF21,NM_019113.4,630,209
4,GDF15,NM_004864.4,927,308
5,VEGFA,NM_001317010.2,642,213
6,FNDC5,NM_001441683.1,783,260
7,METRNL,NM_001363853.2,690,229


## 2. Generator: synonymous codon design

In [5]:
tbl = CodonTable.unambiguous_dna_by_id[1]
syn = defaultdict(list)
for c, a in tbl.forward_table.items(): syn[a].append(c)
syn["*"] = list(tbl.stop_codons)

# codon usage is estimated from the seed panel itself (+1 pseudocount)
counts = Counter()
for v in human.values():
    s = v["cds"]; counts.update(s[i:i+3] for i in range(0, len(s) - 2, 3))
usage = {}
for aa, cods in syn.items():
    tot = sum(counts[c] + 1 for c in cods)
    usage[aa] = {c: (counts[c] + 1) / tot for c in cods}
w_rel = {aa: {c: f / max(d.values()) for c, f in d.items()} for aa, d in usage.items()}

FORBID = ["GGTCTC", "GAGACC", "CGTCTC", "GAGACG", "AATAAA", "TTTTT", "AAAAA", "GGGGG", "CCCCC"]
def gc(s): return (s.count("G") + s.count("C")) / len(s)
def longest_run(s): return max(len(list(g)) for _, g in groupby(s))
def has_forbidden(s): return any(m in s for m in FORBID)
def cai(s):
    ws = [w_rel[tbl.forward_table.get(s[i:i+3], "*")][s[i:i+3]] for i in range(0, len(s) - 2, 3)]
    return float(np.exp(np.mean(np.log(ws))))

def design(cds, T=1.5, gc_range=(0.45, 0.60), tries=300, seed=0):
    rng = random.Random(seed)
    prot = [tbl.forward_table.get(cds[i:i+3], "*") for i in range(0, len(cds) - 2, 3)]
    best = None
    for _ in range(tries):
        out = []
        for k, aa in enumerate(prot):
            if k == 0 or k == len(prot) - 1: out.append(cds[3*k:3*k+3]); continue  # keep start/stop
            cods = syn[aa]; p = np.array([usage[aa][c] ** (1 / T) for c in cods]); p /= p.sum()
            out.append(rng.choices(cods, weights=p)[0])
        s = "".join(out)
        if gc_range[0] <= gc(s) <= gc_range[1] and longest_run(s) < 6 and not has_forbidden(s):
            return s
        best = best or s
    return best

## 3. Checks against NCBI, UniProt and the ExerkineAtlas

In [6]:
def fold_metrics(s, window=60):
    _, mfe = RNA.fold(s.replace("T", "U"))
    _, mfe5 = RNA.fold(s[:window].replace("T", "U"))
    return dict(mfe=mfe, mfe_5p=mfe5)

def blastn_top(s):
    # slow (minutes); run on a few top designs only
    rec = NCBIXML.read(NCBIWWW.qblast("blastn", "refseq_rna", s, hitlist_size=3, megablast=False))
    if not rec.alignments: return dict(blast_top="no hit", blast_id=0.0, blast_cov=0.0)
    a = rec.alignments[0]; h = a.hsps[0]
    return dict(blast_top=a.hit_def[:70], blast_id=h.identities / h.align_length, blast_cov=h.align_length / len(s))

ATLAS_PATH = None   # e.g. "exerkine_atlas.csv"
ATLAS_GENE_COL = "gene"
atlas = pd.read_csv(ATLAS_PATH, sep=None, engine="python") if ATLAS_PATH else None
def atlas_row(gene):
    if atlas is None: return {"in_atlas": None}
    hit = atlas[atlas[ATLAS_GENE_COL].astype(str).str.upper() == gene.upper()]
    return {"in_atlas": len(hit) > 0, "atlas_rows": len(hit)}

# optional: Nucleotide Transformer masked log-likelihood (large download)
USE_NT_LM = False
if USE_NT_LM:
    import torch
    from transformers import AutoTokenizer, EsmForMaskedLM
    NT = "InstaDeepAI/nucleotide-transformer-500m-human-ref"
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    nt_tok = AutoTokenizer.from_pretrained(NT); nt_lm = EsmForMaskedLM.from_pretrained(NT).to(dev).eval()
    @torch.no_grad()
    def nt_loglik(s, frac=0.15, seed=0):
        ids = nt_tok(s, return_tensors="pt", truncation=True, max_length=nt_tok.model_max_length)["input_ids"].to(dev)
        special = torch.tensor(nt_tok.all_special_ids, device=dev)
        cand = (~torch.isin(ids[0], special)).nonzero().squeeze(-1).tolist()
        pos = random.Random(seed).sample(cand, max(1, int(frac * len(cand))))
        m = ids.clone(); m[0, pos] = nt_tok.mask_token_id
        lp = torch.log_softmax(nt_lm(m).logits[0, pos], -1)
        return lp.gather(1, ids[0, pos].unsqueeze(1)).mean().item()

## 4. Run the gym

In [7]:
rows = []
for gene, seed in human.items():
    parent, prot = seed["cds"], seed["protein"]
    up_acc, up_prot = uniprot_protein(gene)
    rat_cds = (rat.get(gene) or {}).get("cds")
    for i in range(20):
        v = design(parent, T=1.5, seed=i)
        tr = str(Seq(v).translate(to_stop=True))
        row = dict(gene=gene, design=i, cds=v, len_nt=len(v),
                   translates_ok=(tr == prot.rstrip("*")),
                   uniprot_id=identity(tr, up_prot, aa_aln) if up_prot else np.nan,
                   id_human_cds=identity(v, parent, nt_aln),
                   id_rat_cds=identity(v, rat_cds, nt_aln) if rat_cds else np.nan,
                   gc=gc(v), cai=cai(v), run=longest_run(v), forbidden=has_forbidden(v),
                   **fold_metrics(v), **atlas_row(gene))
        if USE_NT_LM: row["nt_ll"] = nt_loglik(v)
        rows.append(row)
df = pd.DataFrame(rows)
df["pass"] = df.translates_ok & ~df.forbidden & (df.run < 6) & df.gc.between(0.45, 0.60)
df.drop(columns="cds").groupby("gene")[["translates_ok", "pass", "id_human_cds", "gc", "cai", "mfe_5p"]].mean()

,translates_ok,pass,id_human_cds,gc,cai,mfe_5p
gene,,,,,,
BDNF,1.0,0.60,0.761850,0.526996,0.674487,-12.660
FGF21,1.0,0.65,0.754365,0.595079,0.634746,-15.910
FNDC5,1.0,0.35,0.774840,0.576373,0.668541,-27.385
GDF15,1.0,0.80,0.743528,0.596117,0.626317,-14.455
IL15,1.0,1.00,0.737745,0.472304,0.694267,-15.080
IL6,1.0,1.00,0.759912,0.492193,0.662336,-11.160
METRNL,1.0,1.00,0.769348,0.582899,0.653205,-8.764
VEGFA,0.0,0.00,0.775312,0.529206,0.678093,-10.050


In [8]:
# Rank passing designs; BLASTn only the best per gene (slow)
ranked = (df[df["pass"]].sort_values(["gene", "cai", "mfe_5p"], ascending=[True, False, False])
            .groupby("gene").head(1).reset_index(drop=True))
ranked = pd.concat([ranked, pd.DataFrame([blastn_top(s) for s in ranked.cds])], axis=1)
ranked.drop(columns="cds")

,gene,design,len_nt,translates_ok,uniprot_id,id_human_cds,id_rat_cds,gc,cai,run,forbidden,mfe,mfe_5p,in_atlas,pass,blast_top,blast_id,blast_cov
0,BDNF,7,789,True,1.000000,0.783270,0.770596,0.542459,0.717339,4,False,-286.799988,-13.200000,None,True,PREDICTED: Ceratotherium simum simum brain der...,0.795039,0.970849
1,FGF21,0,630,True,1.000000,0.776190,0.681021,0.600000,0.658269,4,False,-234.000000,-18.200001,None,True,PREDICTED: Pongo abelii fibroblast growth fact...,0.779365,1.000000
2,FNDC5,7,783,True,1.000000,0.791826,0.792063,0.593870,0.694511,4,False,-296.600006,-40.500000,None,True,Homo sapiens fibronectin type III domain conta...,0.798949,0.971903
3,GDF15,17,927,True,1.000000,0.743258,0.584430,0.596548,0.635496,4,False,-352.500000,-13.100000,None,True,Homo sapiens growth differentiation factor 15 ...,0.743258,1.000000
4,IL15,18,408,True,0.933333,0.745098,0.671569,0.487745,0.750940,4,False,-115.000000,-15.700000,None,True,PREDICTED: Pan troglodytes interleukin 15 (IL1...,0.748130,0.982843
5,IL6,19,570,True,0.994709,0.771930,0.584211,0.500000,0.702192,4,False,-156.199997,-11.300000,None,True,PREDICTED: Colobus angolensis palliatus interl...,0.783486,0.956140
6,METRNL,10,690,True,1.000000,0.775362,0.681159,0.592754,0.683238,4,False,-278.799988,-10.800000,None,True,PREDICTED: Gorilla gorilla gorilla meteorin li...,0.776812,1.000000


In [9]:
# Export ranked designs (RNA alphabet) for downstream tokenization
with open("exerkine_rna_designs.fasta", "w") as f:
    for _, r in ranked.iterrows():
        f.write(f">{r.gene}|design{r.design}|cai={r.cai:.2f}|gc={r.gc:.2f}\n{r.cds.replace('T', 'U')}\n")

**Reading the table:** `translates_ok` must be True and `uniprot_id` ~1.0 (the protein is unchanged). `id_human_cds` well below 1.0 means the nucleotides diverge from the natural CDS; `blast_id`/`blast_cov` show how close the nearest RefSeq RNA still is, and `blast_top` should still name the parent gene. Higher `cai` and a less negative `mfe_5p` favor expression. `in_atlas` confirms the parent is an annotated exerkine.

**Limits:** designs are CDS only (no UTRs), codon usage comes from the seed panel rather than a host-genome table, and CAI/MFE are proxies, not expression measurements.